# Homework #2 (GR5069)

Sarayu Rao

In [0]:
#Imports
from pyspark.sql.functions import col, round, avg, upper, substring, when, length, floor, datediff, current_date, max, min, sum, when
import pyspark.sql.functions as F
from pyspark.sql import Window


**Q1**   
My logic for this question was to use the pitstops data set, and group by both raceID and driverID to calculate the average milliseconds spent at the pitstop. 

In [0]:
#Load and display pitstop dataset
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
display(df_pitstops)

In [0]:
#Cast necessary columns to integers from strings 
df_pitstops = df_pitstops.withColumns({"raceId": col("raceId").cast("int"),
                                      "driverId": col("driverId").cast("int"),
                                      "stop": col("stop").cast("int"),
                                      "lap": col("lap").cast("int"),
                                      "milliseconds": col("milliseconds").cast("int")})

In [0]:
#Calculate average pitstop time for each driver
df_avg_pit = df_pitstops.groupBy("raceId","driverId").avg("milliseconds")
df_avg_pit = df_avg_pit.orderBy("raceId")
display(df_avg_pit)

In [0]:
#Calculate max and min pitstop time for each race
df_max_pit = df_pitstops.groupBy("raceId").max('milliseconds')
df_min_pit = df_pitstops.groupBy("raceId").min('milliseconds')

#Join max and min pitstop datasets
df_max_min_pit = df_max_pit.join(df_min_pit, on="raceId")
df_max_min_pit = df_max_min_pit.withColumnsRenamed({"max(milliseconds)": "Slowest Pit", "min(milliseconds)": "Fastest Pit"})
display(df_max_min_pit)

Q1 Code explanation: First I load the pitstop dataset, and cast columns I know I'll be utilizing from strings to integers using the .cast() function. Next I use the groupBy function and the avg function to get the average pitstop time for each driver for each race. By including both raceId and driverId in the groupBy function, it automatically calculated the average pitstop time for each driver for each race.  

To calculate the max and min pitstop time, I created 2 dataframes (one groups by raceId to get the max pitstop time, the other one is for min). Then I join the 2 datasets on raceId to get one dataset that show both slowest and fastest pitstop times for each race.

**Q2**  
My logic for Q2 was to use the average pitstop dataset created in Q1 that already has average pitstop time for each driver per race. I will then get the position order for each driver for each race, and sort each race by the driver's position. I use positionOrder specifically because it provides a position for drivers who didn't finish the race. 

In [0]:
#Load and display results dataset
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
display(df_results)

In [0]:
#Cast necessary columns to integers from strings
df_results = df_results.withColumns({"raceId": col("raceId").cast("int"),
                                      "driverId": col("driverId").cast("int"),
                                      "resultId": col("resultId").cast("int"),
                                      "positionOrder": col("positionOrder").cast("int"),
                                     })

In [0]:
#Join pitstop and results datasets to get positionOrder for each driver
df_race_results = df_results["raceId","driverId","positionOrder"]
df_sorted_pit = df_avg_pit.join(df_race_results, on:= ["raceId","driverId"])

#Sort each race by positionOrder
df_sorted_pit = df_sorted_pit.orderBy("raceId","positionOrder")
display(df_sorted_pit)

Q2 Code explanation: After loading the results dataset, I casted the necessary columns from strings to integers. Next I selected racedId, driverId, and positionOrder from the results table and join it on raceId and driverID in the average pit dataset from Q1. By joining on both raceId and driverId, it joins on each unique pair of those 2 IDs. Then I sort by raceId and positionOrder, which sorts each raceId by positionOrder. 

**Q3**  
My logic for Q3 was to create new driver codes for all the drivers. Then I check if the code column in the drivers table is null ('\N'), and if yes, fill it in with the created driver code, if not null, leave it as is.  
Creating driver code logic: The way I created each driver code was to check if the driverRef had an underscore in it. For driverRefs that had no "_", I capitalized the frist 3 chars to use as the diver code. For refs that had an underscore, I checked if the number of chars after the underscore was less than 3, as this meant that it was usually jr or sr and not a last name, and took the first 3 chars from before the underscore. For all others, I took the first 3 chars from after the underscore.  
Some drivers will have the same code. This is because not all drivers are on the grid at the same time, so codes can overlap. Additionally, if drivers did have the same code at the same time, each team would create a custom code to then get approved, making it hard to apply that logic in my code.  

In [0]:
#Load and display drivers dataset
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
display(df_drivers)

In [0]:
#Check that "\N" is a string and not null
print(df_drivers.filter(col("code") == "\\N").count()   )
df_drivers.filter(col("code").isNull()).count()   

In [0]:
#Create a new driver code for all drivers using logic mentioned above
driver_code = when(~(col("driverRef").contains("_")), F.upper(substring("driverRef", 1, 3))).when((col("driverRef").contains("_")) & (length(F.substring_index("driverRef","_",-1))< 3), F.upper(substring(F.substring_index("driverRef", "_", 1), 1, 3))).otherwise(F.upper(substring(F.substring_index("driverRef", "_", -1), 1, 3)))

#Check if code column has "\N" and fill it in with the created driver_code
df_drivers = df_drivers.withColumn("code", 
                                   when((col("code")=="\\N"), driver_code)
                                   .otherwise(col("code")))


In [0]:
#Checking results
display(df_drivers)

Q3 code explanation: After loading the drivers dataset, I check to make sure that the "\N" is a string and not null.  
driver_code code: I use the .withColumn function to create a new column of the new driver_codes, and using .where to create new codes given certain conditions. The first condition uses .contains and ~ to check that driverRef doesn't contain an underscore. The ~ is equivaled to an ! (checking that some is NOT true). Then I get the substring of the first 3 chars of driverRef using substring, then I use upper to make them all uppercase. The next condition checks if driverRef as an underscore and uses .substring_index and length() to check whether the number of chars after the underscore is less than 3. If this condition is met, I take the substring of the first 3 chars before the underscore and capitalize it. The .otherwise is the last condition if the others aren't met, which takes the substring of the first 3 chars after the underscore and capitalizes it.

**Q4**  
My logic for Q4 was to first get the age of each driver during each race. This involved adding the drivers dob, forename, and surname to the results table. I also had to get the date of each race from the races dataset. I then calculated age by using the datediff function to get the age of each driver during a race by subtracting the driver's dob from the race date and dividing by 365.   
To get the the youngest and oldest driver of each race, I. created a separate table that grouped each race by the oldest and youngest age. Then create 2 tables (one for oldest and one for youngest), which gets the driverId and name of the oldest/youngest driver. I then join the 2 tables to create 1 table showing the oldest and youngest driver for each race. 

In [0]:
#Join driverId, dob, forename, and surname from drivers to results
df_results = df_drivers.select("driverId", "dob", "forename", "surname").join(df_results, on=["driverId"])
display(df_results)

In [0]:
#Load and display races dataset
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
display(df_races)

In [0]:
#Join raceId and date from races to results
df_results = df_races.select("raceId", "date").join(df_results, on=["raceId"])
display(df_results)

In [0]:
#Get the age of each driver for each race
df_results = df_results.withColumn("age", round(datediff(col("date"), col("dob"))/365,2))
display(df_results)

In [0]:
#Get oldest/youngest age per race
df_ages = df_results.groupBy("raceId").agg(max("age").alias("oldest_age"), min("age").alias("youngest_age"))

#Join back to results to get the driverId for each oldest/youngest age
#Get oldest driver
df_oldest = df_ages.join(df_results, (df_ages.raceId == df_results.raceId) & (df_ages.oldest_age == df_results.age)).select(df_ages.raceId, "oldest_age", df_results.driverId.alias("oldest_driverId"), df_results.forename.alias("oldest_driverName"), df_results.surname)

#Get youngest driver            
df_youngest = df_ages.join(df_results, (df_ages.raceId == df_results.raceId) & (df_ages.youngest_age == df_results.age)).select(df_ages.raceId, "youngest_age", df_results.driverId.alias("youngest_driverId"),df_results.forename.alias("youngest_driverName"), df_results.surname)

#Join tables together
df_old_young_race = df_oldest.join(df_youngest, on="raceId")
display(df_old_young_race)

Q4 Code explanation: I select the driverId, dob, forename, and surname columns from the drivers table to join on driverId in the results table. I then load in the races dataset to select the raceId and date, and join on raceId in the results table. Now that the results table has each drivers dob and the date of the race, I use the datediff function to calculate the age of each driver. I round to 2 decimal places so that when creating the oldest and youngest driver, getting the max and min won't have any duplicates if drivers have the same age in years.  
To create the youngest and oldest drivers, I create the df_ages dataset by groupingby racedId and getting the aggregate max and min ages using .agg(). I use .alias to rename the aggregate columns for easier referencing in my later code. I create a table for oldest drivers by finding the row from results where the age in from the oldestage column in the df_ages dataset is the same for that raceId, then for that row I get the driverId, forename, and surname. I do the same to create the youngest drivers dataset. I then join the 2 datasets on raceId to get the oldest and youngest drivers for each race. 

**Q5**  
My logic for Q5 was for each driver in a race, to get the cumulative number of podiums from all races before that race. 

In [0]:
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
display(df_races)

In [0]:
#Create w to be able to keep all driver rows and not collapse when aggregating
w = Window.partitionBy("driverId").orderBy("date").rowsBetween(Window.unboundedPreceding, -1)

#Get cumulative podium results 
df_podiums = df_results.withColumn("cum_wins",
            
        sum(when(col("positionOrder") == 1, 1).otherwise(0)).over(w)).withColumn("cum_2nd",sum(when(col("positionOrder") == 2, 1).otherwise(0)).over(w)).withColumn("cum_3rd",sum(when(col("positionOrder") == 3, 1).otherwise(0)).over(w))

display(df_podiums)

Q5 code explanation: I use the Windows function to be able to group by driverID without it all collapsing in to one row. To breakdwon the Window function, I partionBy driverId which groups all driverIds separately but keeps all the rows. I order by the date of each race for each driver. The rowsBetween(Window.unboundPreceeding, -1) function says to aggregate from the first row of the driver all the way to the row right before the current row. This means I now have grouped each driverId together, and for all driverId, ordered their rows in order of the date of their races. When I aggregate over these driverIds, for each row, I'll look at the first row ever all the way to the row before this.  
Then I create 3 columns that look at the positionOrder and sum the number of 1, 2, 3 positions for each driver over the window w created. The when() function creates a placeholder column to allocate 1 for each win, then we sum nd 3rd place as well. 

**Q6**  
I wanted to see which driver had the fastest lap time for each race. I used logic similar to question 4 to first get the fastest lap for each race, then added the driver who what that laptime for that race.

In [0]:
#Get fastest lap for each race
df_fastest_lap = df_results.filter(col("fastestLapTime") != "\\N").groupBy("raceId").agg(min('fastestLapTime').alias('fastestLap'))

#Join back to results to get the driverId and name for each race's fastest lap
df_fastest_driver = df_fastest_lap.join(df_results, (df_results.raceId == df_fastest_lap.raceId) & (df_results.fastestLapTime == df_fastest_lap.fastestLap)).select(df_fastest_lap.raceId, df_fastest_lap.fastestLap, df_results.driverId.alias("fastest_driverId"), df_results.forename.alias("fastest_driverName"), df_results.surname)
display(df_fastest_driver)

Q6 code explanation: I first filter out the"\N" vales from the fastestLapTime column, then groupBy raceID and use the agg() function to get the min fastestLapTime. Then I join the results table by matching the raceId and fastestLapTime, and select the driverID, forename, and surname. 